# Tokenizer: one artifact, declared and then bound

`tokenizers.bpe` is one tokenizer family, whole: the artifact, the jobs that
build it, and the BPE internals both share. `Tokenizer` is a dag Artifact and
the tokenizer itself, in one object:

- as an **artifact**, it's the parameters of a training run (vocab_size,
  special_tokens, which sources) plus the folder it owns. You can write one
  down, hash it, declare it, and put it in a manifest before anything has
  been trained.
- as a **tokenizer**, it encodes and decodes text -- but only once it holds
  the vocab and merges a finished run produced. Those aren't parameters (they
  are what the job *wrote*), so they arrive through `bind(root)`, which reads
  them back out of the folder.

This notebook walks the whole path -- declare, build, bind, encode --
against a local folder, with no Modal involved.

In [1]:
import logging
from pathlib import Path
from types import SimpleNamespace

from dag import resolve as dag_resolve
from sources.artifact import Source
from tokenizers.bpe import TokenizedSource, Tokenizer

# A throwaway stand-in for the Modal volume. On Modal, root is Path(STORAGE)
# and jobs are driven by main.py's run_job; nothing in the artifact/job layer
# knows about either -- a job takes a root and a worker, and that's all.
ROOT = Path(".scratch/demo-volume").resolve()
ROOT.mkdir(parents=True, exist_ok=True)

# A job only ever reaches for worker.log, so a namespace with a logger stands
# in for the real runtime.Worker (which carries a lease and a call id it has
# nothing to lease against here).
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
worker = SimpleNamespace(log=logging.getLogger("demo"))

ROOT

PosixPath('/Users/oguz/Projects/launchpad/.scratch/demo-volume')

## 1. Declaring one

Written by hand, same as in [declare.ipynb](declare.ipynb). It answers
*which* tokenizer this is and *where* it lives -- and nothing about bytes,
because nothing has been trained yet.

In [2]:
romeojuliet = Source(
    name="romeojuliet", url="https://www.gutenberg.org/cache/epub/1513/pg1513.txt"
)

tokenizer = Tokenizer(
    vocab_size=1000,
    special_tokens=("<|endoftext|>",),
    sources=(romeojuliet,),
)

print(tokenizer.uid)  # readable id, derived from the parameters
print(tokenizer.artifact_path)  # the folder it owns, relative to root
print(tokenizer.paths(ROOT)["tokenizer"])  # where its one file will land
print(tokenizer.exists(ROOT))  # nothing built yet

bpe-1000-feeeeefa90
tokenizers/bpe-1000-feeeeefa90
/Users/oguz/Projects/launchpad/.scratch/demo-volume/tokenizers/bpe-1000-feeeeefa90/tokenizer.json
False


In [ ]:
# So it can't tokenize anything yet: there is no trained state to bind, and
# nothing to read it from. Both ways of asking say so rather than handing back
# something half-formed.
try:
    tokenizer.encode("But soft")
except RuntimeError as err:
    print("not bound ->", err)

try:
    tokenizer.bind(ROOT)
except FileNotFoundError as err:
    print("not built ->", err)

## 2. Declare, then build

`TokenizedSource` is asked for rather than the tokenizer itself, to show the
tokenizer being used as a dependency: it's romeojuliet encoded *by this
tokenizer*, so resolving it pulls in the source and the tokenizer beneath it.

In [4]:
tokenized = TokenizedSource(tokenizer=tokenizer, source=romeojuliet)

declaration = dag_resolve.Declaration(tokenized, ROOT).check()
declaration  # reads the disk, writes nothing

run - under /Users/oguz/Projects/launchpad/.scratch/demo-volume
  new        sources/romeojuliet
  new        tokenizers/bpe-1000-feeeeefa90
  new        tokenizers/bpe-1000-feeeeefa90/bin/romeojuliet

3 new
ok -- 3 to declare

In [5]:
declaration.write()  # one manifest.json per artifact, dependencies first

[PosixPath('/Users/oguz/Projects/launchpad/.scratch/demo-volume/sources/romeojuliet/manifest.json'),
 PosixPath('/Users/oguz/Projects/launchpad/.scratch/demo-volume/tokenizers/bpe-1000-feeeeefa90/manifest.json'),
 PosixPath('/Users/oguz/Projects/launchpad/.scratch/demo-volume/tokenizers/bpe-1000-feeeeefa90/bin/romeojuliet/manifest.json')]

In [6]:
# Run the plan in dependency order: download the source, train the tokenizer,
# then encode the source with it. Re-running the cell skips what's done, so
# only the missing pieces are rebuilt.
for job in dag_resolve.job_list(dag_resolve.resolve(tokenized)):
    if dag_resolve.status(job.artifact, ROOT) == "done":
        print(f"already done: {job.artifact.artifact_path}")
        continue
    job.run(ROOT, worker)

downloading romeojuliet from https://www.gutenberg.org/cache/epub/1513/pg1513.txt


wrote 167469 chars for romeojuliet


training BPE tokenizer (vocab_size=1000) on 1 source(s)


pretokenizing and building frequency map:   0%|          | 0.00/170k [00:00<?, ?B/s]

pretokenizing and building frequency map: 100%|██████████| 170k/170k [00:00<00:00, 2.81MB/s]

...

merging pairs:  99%|█████████▊| 732/743 [00:01<00:00, 719.49it/s]

merging pairs: 100%|██████████| 743/743 [00:01<00:00, 658.54it/s]


trained, vocab has 1000 entries


tokenizing romeojuliet


wrote 65754 tokens for romeojuliet


## 3. Binding the trained state

The folder is filled in now, so `bind(root)` can hand back a Tokenizer with
its vocab and merges loaded. It's a *new* object, equal to the one you
declared (same parameters, same `==`) but never the same object -- the
recipe you called `bind` on stays exactly as unbound as it was. So the
convention is to reassign: `tokenizer = tokenizer.bind(root)`, then use
`tokenizer` from there on.

In [ ]:
tokenizer = tokenizer.bind(ROOT)  # a new, bound Tokenizer -- reassign, don't discard

print(f"{len(tokenizer.vocab)} vocab entries, {len(tokenizer.merges)} merges")
print("first merges:", [a + b for a, b in tokenizer.merges[:8]])
print("specials:", tokenizer.special_tokens)  # a parameter, not something loaded

# the same tokenizer, spelled out again: still equal, still the same folder
print(Tokenizer(vocab_size=1000, special_tokens=("<|endoftext|>",), sources=(romeojuliet,)) == tokenizer)

In [ ]:
line = "But soft, what light through yonder window breaks?<|endoftext|>"

ids = tokenizer.encode(line)
print(ids)
print([tokenizer.vocab[i] for i in ids])  # what each id stands for
print(repr(tokenizer.decode(ids)))  # round-trips, special token included

The same object read the other way round: `TokenizedSource` holds this
source already encoded, and the tokenizer that encoded it is what turns those
ids back into text.

In [ ]:
from array import array

token_ids = array("H")
token_ids.frombytes(tokenized.paths(ROOT)["tokens"].read_bytes())

print(f"{len(token_ids)} tokens on disk at {tokenized.artifact_path}")
print(repr(tokenizer.decode(token_ids[:80])))

## The other direction

`bind(root)` is the only way this artifact ever gets its vocab and merges --
it reads them off disk, and that's it. Writing the file in the first place
isn't the artifact's job: it's `TokenizerJob.save`'s, called from the last
line of `TokenizerJob.run`, a few classes down the same module
([tokenizers/bpe.py](../tokenizers/bpe.py)):

```python
self.save(root, vocab, merges)
```

So a tokenizer's life is three calls, split across two objects:

| | call | who does it |
| --- | --- | --- |
| trained state -> its folder | `TokenizerJob(tokenizer).save(root, vocab, merges)` | the training job, once |
| its folder -> trained state | `tokenizer.bind(root)` | everyone downstream |
| use it | `tokenizer.encode(...)` / `tokenizer.decode(...)` | anyone, once bound |

The job writes into the folder the artifact owns but never touches the
artifact's own state -- it already has `vocab`/`merges` in hand from training,
so it has no reason to. `bind(root)` is what turns a written file back into a
usable object: it checks the file agrees with what this artifact declares --
a tokenizer.json whose special tokens aren't these raises instead of quietly
encoding with someone else's vocab.

Nothing above changes on Modal: `ROOT` becomes `Path(STORAGE)`, `worker`
becomes the real `runtime.Worker`, and the calls read the same.

In [10]:
# Cleanup, if you want the demo volume gone:
# import shutil; shutil.rmtree(ROOT)